# 03 - A/B experiment (causal module)
Goal: show how to measure whether a new curing method really raises strength, using a simulated randomised experiment on real specimens.

- Treatment: new curing method, true effect +2 MPa (assumed, because the UCI data is observational).
- Outcome: measured strength. CUPED covariate: the out-of-fold model prediction (it only uses the mix design, so the curing method cannot change it).
- All statistics come from `src/experiment.py`; the specimens come from SQL (`experiment_pool` query).

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from db.queries import run_query
from src.experiment import (
    achieved_power,
    analyze_experiment,
    cuped_sd,
    power_sample_size,
    simulate_experiment,
)

pool = run_query('experiment_pool')
sd = pool['strength_mpa'].std(ddof=1)
rho = pool['strength_mpa'].corr(pool['predicted_mpa'])
print(f'pool: {len(pool)} specimens | sd = {sd:.2f} MPa | corr(strength, prediction) = {rho:.3f}')
print(f'sd left after CUPED: {cuped_sd(sd, rho):.2f} MPa')

## Step 1 - Power analysis
How many specimens per arm do we need to detect a given uplift with 80% power? CUPED removes the noise the model already explains, so it needs far fewer. The pool can supply at most 399 per arm.

In [ ]:
mdes = np.arange(1.0, 5.01, 0.25)
naive_n = [power_sample_size(sd, m) for m in mdes]
cuped_n = [power_sample_size(cuped_sd(sd, rho), m) for m in mdes]

plt.figure(figsize=(7, 4))
plt.plot(mdes, naive_n, marker='o', label='naive')
plt.plot(mdes, cuped_n, marker='o', label='CUPED')
plt.axhline(len(pool) // 2, color='gray', linestyle='--', label='max per arm in pool')
plt.yscale('log')
plt.xlabel('minimum detectable effect (MPa)')
plt.ylabel('specimens per arm (log scale)')
plt.title('Sample size needed for 80% power')
plt.legend()
plt.show()

## Step 2 - One experiment
Randomise 150 specimens per arm, then analyse the same data two ways. Both confidence intervals should contain the true +2 MPa; the CUPED one is much narrower.

In [ ]:
rng = np.random.default_rng(7)
sim = simulate_experiment(pool, n_per_arm=150, true_effect=2.0, rng=rng)
res = analyze_experiment(sim['y'], sim['treatment'], sim['x'])
print(pd.DataFrame(res).T[['effect', 'ci_low', 'ci_high', 'p_value']].round(3))

plt.figure(figsize=(7, 3))
for i, method in enumerate(['naive', 'cuped']):
    r = res[method]
    plt.errorbar(r['effect'], i, xerr=[[r['effect'] - r['ci_low']], [r['ci_high'] - r['effect']]], fmt='o', capsize=5)
plt.axvline(2.0, color='green', linestyle='--', label='true effect')
plt.axvline(0.0, color='gray', linestyle=':', label='no effect')
plt.yticks([0, 1], ['naive', 'CUPED'])
plt.xlabel('estimated uplift (MPa) with 95% CI')
plt.legend()
plt.show()

## Step 3 - Repeat the experiment 500 times
One experiment can be lucky or unlucky. Repeating it shows the spread of each estimator: both are centred on +2 (unbiased), but CUPED is far tighter.

In [ ]:
rng = np.random.default_rng(42)
estimates = {'naive': [], 'cuped': []}
for _ in range(500):
    s = simulate_experiment(pool, n_per_arm=150, true_effect=2.0, rng=rng)
    r = analyze_experiment(s['y'], s['treatment'], s['x'])
    for method in estimates:
        estimates[method].append(r[method]['effect'])

plt.figure(figsize=(7, 4))
plt.hist(estimates['naive'], bins=30, alpha=0.6, label='naive')
plt.hist(estimates['cuped'], bins=30, alpha=0.6, label='CUPED')
plt.axvline(2.0, color='green', linestyle='--', label='true effect')
plt.xlabel('estimated uplift (MPa)')
plt.ylabel('number of experiments')
plt.legend()
plt.show()

for method, values in estimates.items():
    print(f'{method}: mean = {np.mean(values):.3f} | sd = {np.std(values, ddof=1):.3f}')

## Step 4 - Power vs sample size
Probability of detecting the +2 MPa effect as the experiment grows.

In [ ]:
ns = np.arange(10, 400, 10)
plt.figure(figsize=(7, 4))
plt.plot(ns, [achieved_power(sd, n, 2.0) for n in ns], label='naive')
plt.plot(ns, [achieved_power(cuped_sd(sd, rho), n, 2.0) for n in ns], label='CUPED')
plt.axhline(0.8, color='gray', linestyle='--', label='80% power')
plt.xlabel('specimens per arm')
plt.ylabel('power')
plt.title('Power to detect +2 MPa')
plt.legend()
plt.show()

## Summary
- Pool: 799 specimens, sd 15.8 MPa, corr(strength, prediction) 0.938.
- To detect +2 MPa at 80% power: 981 specimens per arm naive vs 118 with CUPED (88% fewer).
- 500 simulated experiments at 150 per arm: power 18% naive vs 89% CUPED; false-positive rate close to 5% (6.4% / 3.8%); CI coverage close to 95% (93.6% / 96.4%).
- Limitations: the +2 MPa effect is simulated; randomisation is per specimen, so batch effects are ignored.